# Literature RAG Integration Test

Integration tests for the 3 RAG tools in `literature_rag.py`:
- `prepare_rag_literature` — download papers, index to Vector DB
- `search_literature_fulltext` — semantic/hybrid search over indexed papers
- `validate_claim_fulltext` — validate biological claims against full-text

### Prerequisites
- CCI-RAG MCP server configured (`CCI_RAG_PYTHON`, `CCI_RAG_SERVER`)
- pubmed-download MCP server configured (`PUBMED_MCP_PYTHON`, `PUBMED_MCP_SERVER`)
- `.env` file with required API keys (`PUBMED_EMAIL`, `JINA_API_KEY`, `NCBI_API_KEY`)
- Docker running (langconnect-rag backend)

In [1]:
import json
import os
import sys
import tempfile
import time
import shutil
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Imports — public API
from src.agents.deep_interpreter.tools.literature_rag import (
    # @tool sync wrappers
    prepare_rag_literature,
    search_literature_fulltext,
    validate_claim_fulltext,
    # Async internals
    prepare_rag_for_interpretation,
    search_fulltext_rag,
    validate_claim_with_rag,
    check_rag_availability,
    check_rag_availability_sync,
    download_papers_for_rag,
    index_papers_to_vectordb,
    # State management
    _RAG_STATE,
    get_rag_status,
    is_rag_mode_active,
    reset_rag_state,
    RAG_COLLECTION_PREFIX,
)
from src.agents.deep_interpreter.tools.mcp_clients import (
    run_async,
    invalidate_client,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST
Python: /Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST/.venv/bin/python


In [2]:
import logging

# Configure logging — DEBUG for literature_rag, WARNING for others
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("src.agents.deep_interpreter.tools.literature_rag").setLevel(logging.DEBUG)
logging.getLogger("src.agents.deep_interpreter.tools.mcp_clients").setLevel(logging.DEBUG)

# Test results tracker
TEST_RESULTS: dict[str, bool] = {}

def record(name: str, passed: bool) -> None:
    """Record a test result and print status."""
    TEST_RESULTS[name] = passed
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {name}")

print("Logging and test tracker ready.")

Logging and test tracker ready.


## Section 2: Infrastructure Check

In [3]:
# Check required environment variables
REQUIRED_VARS = [
    "CCI_RAG_PYTHON", "CCI_RAG_SERVER",
    "PUBMED_MCP_PYTHON", "PUBMED_MCP_SERVER",
    "PUBMED_EMAIL",
]
OPTIONAL_VARS = ["JINA_API_KEY", "NCBI_API_KEY", "CCI_RAG_BASE_URL"]

print("=== Required Environment Variables ===")
all_required_set = True
for var in REQUIRED_VARS:
    val = os.getenv(var)
    if val:
        # Mask sensitive values
        display = val[:20] + "..." if len(val) > 20 else val
        print(f"  {var} = {display}")
    else:
        print(f"  {var} = NOT SET  <-- REQUIRED")
        all_required_set = False

print("\n=== Optional Environment Variables ===")
for var in OPTIONAL_VARS:
    val = os.getenv(var)
    if val:
        display = val[:15] + "..." if len(val) > 15 else val
        print(f"  {var} = {display}")
    else:
        print(f"  {var} = not set")

record("env_vars_set", all_required_set)

if not all_required_set:
    print("\nWARNING: Some required env vars are missing. Tests may fail.")

=== Required Environment Variables ===
  CCI_RAG_PYTHON = /Users/jungj2/Dropbo...
  CCI_RAG_SERVER = /Users/jungj2/Dropbo...
  PUBMED_MCP_PYTHON = /Users/jungj2/Dropbo...
  PUBMED_MCP_SERVER = /Users/jungj2/Dropbo...
  PUBMED_EMAIL = junghyunjj219@gmail....

=== Optional Environment Variables ===
  JINA_API_KEY = jina_e8b5a2ba6b...
  NCBI_API_KEY = c9553ec3b65d4b0...
  CCI_RAG_BASE_URL = http://localhos...
  [PASS] env_vars_set


In [4]:
# Check RAG availability and list CCI-RAG tools
reset_rag_state()  # Start fresh

rag_available = await check_rag_availability()
print(f"RAG available: {rag_available}")
record("rag_infra_available", rag_available)

if rag_available:
    from src.agents.deep_interpreter.tools.literature_rag import _get_cci_rag_tools
    tools = await _get_cci_rag_tools()
    print(f"\nCCI-RAG tools ({len(tools)}):")
    for t in tools:
        print(f"  - {t.name}")
else:
    print("\nSkipping tool listing — RAG not available.")
    print("Remaining tests will verify fallback behavior only.")

16:10:09 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: using stdio transport
16:10:09 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   CCI_RAG_PYTHON=/Users/jungj2/Dropbox/Metadata/mcp/langconnect_mcp/.venv/bin/python
16:10:09 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   CCI_RAG_SERVER=/Users/jungj2/Dropbox/Metadata/mcp/langconnect_mcp/mcpserver/mcp_server.py
16:10:09 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   CCI_RAG_BASE_URL=http://localhost:8888
16:10:09 [DEBUG] src.agents.deep_interpreter.tools.mcp_clients: MCP client cached for 'CCI-RAG' with 12 tools
16:10:09 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: got 12 tools
16:10:09 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: calling health tool
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: health check result = True
16:10:10 [INFO] src.agent

RAG available: True
  [PASS] rag_infra_available

CCI-RAG tools (12):
  - search_documents
  - list_collections
  - get_collection
  - create_collection
  - delete_collection
  - list_documents
  - add_documents
  - add_documents_from_files
  - delete_document
  - agentic_search
  - get_health_status
  - multi_query


## Section 3: Test Data Creation

In [5]:
# Test isolation: unique part name to avoid collisions with production data
TEST_PART_NAME = "test_rag_check"
TEST_COLLECTION_NAME = f"{RAG_COLLECTION_PREFIX}{TEST_PART_NAME}"

# Real PMIDs with PMC full-text availability
TEST_PMIDS = [
    "37820580",  # ST / HNSCC tumor microenvironment
    "38341519",  # Tumor-associated macrophages / drug resistance
    "36071155",  # Cell-cell interaction in spatial transcriptomics
]

print(f"Test part name: {TEST_PART_NAME}")
print(f"Expected collection: {TEST_COLLECTION_NAME}")
print(f"Test PMIDs: {TEST_PMIDS}")

Test part name: test_rag_check
Expected collection: deep_interpreter_test_rag_check
Test PMIDs: ['37820580', '38341519', '36071155']


In [6]:
# Create test papers JSON in a temp directory
TEST_TEMP_DIR = tempfile.mkdtemp(prefix="rag_test_")
TEST_PAPERS_JSON = os.path.join(TEST_TEMP_DIR, f"{TEST_PART_NAME}_papers.json")

papers_data = [
    {
        "pmid": "37820580",
        "title": "Spatial transcriptomics reveals the tumor microenvironment of HNSCC",
        "year": 2023,
    },
    {
        "pmid": "38341519",
        "title": "Tumor-associated macrophages promote drug resistance via CCL2 signaling",
        "year": 2024,
    },
    {
        "pmid": "36071155",
        "title": "Cell-cell interaction analysis in spatial transcriptomics",
        "year": 2022,
    },
]

with open(TEST_PAPERS_JSON, "w") as f:
    json.dump({"papers": papers_data}, f, indent=2)

print(f"Temp directory: {TEST_TEMP_DIR}")
print(f"Papers JSON: {TEST_PAPERS_JSON}")
print(f"Papers count: {len(papers_data)}")

Temp directory: /var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/rag_test_dokvuyah
Papers JSON: /var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/rag_test_dokvuyah/test_rag_check_papers.json
Papers count: 3


## Section 4: Test 1 — prepare_rag_literature

In [7]:
# Test prepare_rag_for_interpretation (async)
reset_rag_state()

print("Calling prepare_rag_for_interpretation (this downloads papers + indexes)...")
prep_result = await prepare_rag_for_interpretation(TEST_PAPERS_JSON, TEST_PART_NAME)

print(f"\nResult:")
for k, v in prep_result.items():
    print(f"  {k}: {v}")

# Assertions
record("prepare_rag_mode", prep_result.get("rag_mode") is True)
record("prepare_collection_id", prep_result.get("collection_id") is not None)
record("prepare_fulltext_count", prep_result.get("fulltext_count", 0) > 0)
record("prepare_papers_count", prep_result.get("papers_count") == len(TEST_PMIDS))

16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: prepare_rag_for_interpretation START
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: Checking RAG availability...
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: using stdio transport
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   CCI_RAG_PYTHON=/Users/jungj2/Dropbox/Metadata/mcp/langconnect_mcp/.venv/bin/python
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   CCI_RAG_SERVER=/Users/jungj2/Dropbox/Metadata/mcp/langconnect_mcp/mcpserver/mcp_server.py
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   CCI_RAG_BASE_URL=http://localhost:8888
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: got 12 tools
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: calling health tool


Calling prepare_rag_for_interpretation (this downloads papers + indexes)...


16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: check_rag_availability: health check result = True
16:10:10 [INFO] src.agents.deep_interpreter.tools.literature_rag: RAG availability check: True
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: RAG available: True
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: Resolved papers path: /var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/rag_test_dokvuyah/test_rag_check_papers.json -> /private/var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/rag_test_dokvuyah/test_rag_check_papers.json
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: Loading papers from: /private/var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/rag_test_dokvuyah/test_rag_check_papers.json
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: Papers loaded: <class 'list'>, count: 3
16:10:10 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: Extracted PMIDs: 3
16:10:10 [DEBUG] src.agents.d


Result:
  rag_mode: True
  collection_id: a092e3dc-5d93-44d6-a24f-7c9c0a4375ad
  collection_name: test_rag_check
  papers_count: 3
  fulltext_count: 3
  fulltext_ratio: 1.0
  message: RAG ready: 3/3 papers indexed
  [PASS] prepare_rag_mode
  [PASS] prepare_collection_id
  [PASS] prepare_fulltext_count
  [PASS] prepare_papers_count


In [8]:
# Verify _RAG_STATE was updated
print("_RAG_STATE after prepare:")
print(f"  available: {_RAG_STATE['available']}")
print(f"  collection_ids: {_RAG_STATE['collection_ids']}")

record(
    "state_available_true",
    _RAG_STATE["available"] is True,
)
record(
    "state_collection_registered",
    TEST_PART_NAME in _RAG_STATE["collection_ids"],
)

_RAG_STATE after prepare:
  available: True
  collection_ids: {'test_rag_check': 'a092e3dc-5d93-44d6-a24f-7c9c0a4375ad'}
  [PASS] state_available_true
  [PASS] state_collection_registered


In [9]:
# Verify paper files were downloaded
papers_dir = Path("res/papers")
if papers_dir.exists():
    downloaded = list(papers_dir.glob("pmid_*.txt"))
    print(f"Downloaded paper files in {papers_dir}:")
    for p in sorted(downloaded):
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name} ({size_kb:.1f} KB)")

    # Check that at least some of our test PMIDs have files
    test_files = [p for p in downloaded if any(pmid in p.name for pmid in TEST_PMIDS)]
    record("paper_files_exist", len(test_files) > 0)
    print(f"\nTest PMID files found: {len(test_files)}/{len(TEST_PMIDS)}")
else:
    print(f"{papers_dir} does not exist")
    record("paper_files_exist", False)

Downloaded paper files in res/papers:
  pmid_36071155.txt (41.6 KB)
  pmid_37820580.txt (0.7 KB)
  pmid_38341519.txt (74.3 KB)
  [PASS] paper_files_exist

Test PMID files found: 3/3


## Section 5: Test 2 — search_literature_fulltext

In [10]:
# Basic search test
print("Searching: 'tumor microenvironment spatial transcriptomics'...")
search_results = await search_fulltext_rag(
    query="tumor microenvironment spatial transcriptomics",
    collection_name=TEST_PART_NAME,
    limit=5,
)

print(f"\nResults: {len(search_results)}")
for i, r in enumerate(search_results[:3]):
    content_preview = r.get("page_content", "")[:100]
    print(f"  [{i+1}] score={r.get('score', 'N/A'):.3f} | {content_preview}...")

record("search_returns_results", len(search_results) > 0)

# Validate result structure
if search_results:
    first = search_results[0]
    has_keys = all(k in first for k in ["page_content", "metadata", "score"])
    record("search_result_structure", has_keys)
else:
    record("search_result_structure", False)

16:10:11 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: query=tumor microenvironment spatial transcriptomics, collection=test_rag_check
16:10:11 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: Using collection_id: a092e3dc-5d93-44d6-a24f-7c9c0a4375ad
16:10:11 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: claim=tumor microenvironment spatial transcriptomics...
16:10:11 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: calling multi_query tool...


Searching: 'tumor microenvironment spatial transcriptomics'...


16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 5 valid queries
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: spatial transcriptomics in the tumor microenvironment...
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: tumor microenvironment gene expression using spatial transcr...
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: spatially resolved transcriptomics for cancer tumor microenv...
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: analyzing tumor microenvironment with spatial transcriptomic...
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 5: cellular interactions in the tumor microenvironment revealed...
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 5 queries
16:10:15 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_


Results: 5
  [1] score=0.689 | As tumor cells interact with the extracellular interstitium during their growth, they create a parti...
  [2] score=0.669 | At present, the hypothesis of tumor suppressor M1-tumor promoter M2 proposed by Albert Mantovani14is...
  [3] score=0.634 | ### M2-like TAMs modulate the immunological microenvironment facilitating immunosuppressive TME to e...
  [PASS] search_returns_results
  [PASS] search_result_structure


In [11]:
# Multi-query search with diverse biological queries
test_queries = [
    "CCL2 signaling in macrophage recruitment",
    "ligand receptor interaction cell communication",
    "drug resistance mechanisms in cancer",
]

print("Testing diverse queries:")
all_returned_results = True
for q in test_queries:
    results = await search_fulltext_rag(q, TEST_PART_NAME, limit=3)
    status = f"{len(results)} results"
    print(f"  '{q[:50]}...' -> {status}")
    if len(results) == 0:
        all_returned_results = False

record("search_diverse_queries", all_returned_results)

16:10:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: query=CCL2 signaling in macrophage recruitment, collection=test_rag_check
16:10:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: Using collection_id: a092e3dc-5d93-44d6-a24f-7c9c0a4375ad
16:10:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: claim=CCL2 signaling in macrophage recruitment...
16:10:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: calling multi_query tool...


Testing diverse queries:


16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 5 valid queries
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: What is the role of CCL2 (MCP-1) signaling in macrophage rec...
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: How does the CCL2–CCR2 axis regulate monocyte/macrophage rec...
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: Which signaling pathways are activated by CCL2 that promote ...
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: In what biological contexts does CCL2 signaling drive macrop...
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 5: What mechanisms underlie CCL2-driven macrophage recruitment ...
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 5 queries
16:10:32 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: 

  'CCL2 signaling in macrophage recruitment...' -> 3 results


16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 5 valid queries
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: What are the molecular mechanisms of ligand–receptor interac...
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: How does ligand binding to receptors initiate intracellular ...
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: Which ligand–receptor pairs drive paracrine and autocrine si...
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: In immune and nervous systems, how do ligand–receptor intera...
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 5: What experimental and computational approaches are used to s...
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 5 queries
16:10:40 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: 

  'ligand receptor interaction cell communication...' -> 3 results


16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 5 valid queries
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: What are the molecular and cellular mechanisms underlying dr...
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: What roles do cancer stem cells, efflux pumps (e.g., ABC tra...
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: How does the tumor microenvironment contribute to anticancer...
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: What mechanisms drive acquired resistance to targeted cancer...
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 5: What are the major mechanisms of resistance to cancer immuno...
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 5 queries
16:10:54 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: 

  'drug resistance mechanisms in cancer...' -> 3 results
  [PASS] search_diverse_queries


In [12]:
# Search non-existent collection — should return empty list
print("Searching non-existent collection 'nonexistent_xyz'...")
empty_results = await search_fulltext_rag(
    query="anything",
    collection_name="nonexistent_xyz",
    limit=5,
)
print(f"Results: {len(empty_results)} (expected: 0)")
record("search_nonexistent_collection", len(empty_results) == 0)

16:10:56 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: query=anything, collection=nonexistent_xyz
16:10:56 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: Collection nonexistent_xyz not in cache, looking up from DB...
16:10:56 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _lookup_collection_from_db: looking for deep_interpreter_nonexistent_xyz


Searching non-existent collection 'nonexistent_xyz'...


16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _lookup_collection_from_db: raw result type: <class 'list'>
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _lookup_collection_from_db: found 1 collections
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _lookup_collection_from_db: deep_interpreter_nonexistent_xyz NOT FOUND in DB
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: Collection nonexistent_xyz not found in cache or DB
16:10:57 [WARNING] src.agents.deep_interpreter.tools.literature_rag: Collection nonexistent_xyz not indexed


Results: 0 (expected: 0)
  [PASS] search_nonexistent_collection


## Section 6: Test 3 — validate_claim_fulltext

In [13]:
# Test with a matching claim — TAM/drug resistance + relevant PMID
print("Validating matching claim (TAM + drug resistance, PMID 38341519)...")
matching_result = await validate_claim_with_rag(
    claim="Tumor-associated macrophages promote drug resistance through CCL2 signaling in the tumor microenvironment",
    pmid="38341519",
    collection_name=TEST_PART_NAME,
)

print(f"\nResult:")
for k, v in matching_result.items():
    if k == "supporting_text":
        print(f"  {k}: {str(v)[:120]}...")
    else:
        print(f"  {k}: {v}")

# Validate structure
required_keys = {"valid", "confidence", "supporting_text", "source"}
has_structure = required_keys.issubset(matching_result.keys())
record("validate_result_structure", has_structure)

# Confidence should be between 0 and 1
conf = matching_result.get("confidence", -1)
record("validate_confidence_range", 0 <= conf <= 1)

16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: validate_claim_with_rag: claim=Tumor-associated macrophages promote drug resistan..., pmid=38341519, collection=test_rag_check
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: query=Tumor-associated macrophages promote drug resistan..., collection=test_rag_check
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: Using collection_id: a092e3dc-5d93-44d6-a24f-7c9c0a4375ad
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: claim=Tumor-associated macrophages promote drug resistan...
16:10:57 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: calling multi_query tool...


Validating matching claim (TAM + drug resistance, PMID 38341519)...


16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 4 valid queries
16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: What mechanisms enable tumor-associated macrophages to drive...
16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: What is the role of the CCL2-CCR2 axis in TAM-mediated chemo...
16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: How does TAM-derived CCL2 signaling influence therapeutic re...
16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: Are there downstream signaling pathways through which TAM-pr...
16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 4 queries
16:11:04 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: searching query 1/4: What mechanisms enable tumor-associated macrophage...
16:11:04 [DEBUG] src.agents.deep_interpreter.t


Result:
  valid: True
  confidence: 0.781
  supporting_text: ## Targeting M2-like TAMs to overcome tumor resistance...
  source: full_text
  [PASS] validate_result_structure
  [PASS] validate_confidence_range


In [14]:
# Test with a weak/unrelated claim — should have lower confidence
print("Validating weak claim (CRISPR/zebrafish, PMID 38341519)...")
weak_result = await validate_claim_with_rag(
    claim="CRISPR-Cas9 genome editing in zebrafish embryonic development and neural crest migration",
    pmid="38341519",
    collection_name=TEST_PART_NAME,
)

print(f"\nResult:")
for k, v in weak_result.items():
    if k == "supporting_text":
        print(f"  {k}: {str(v)[:120]}...")
    else:
        print(f"  {k}: {v}")

# Weak claim: expect lower confidence than matching claim
matching_conf = matching_result.get("confidence", 0)
weak_conf = weak_result.get("confidence", 1)
print(f"\nMatching confidence: {matching_conf:.3f}")
print(f"Weak confidence: {weak_conf:.3f}")
record("validate_weak_lower_confidence", weak_conf <= matching_conf)

16:11:06 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: validate_claim_with_rag: claim=CRISPR-Cas9 genome editing in zebrafish embryonic ..., pmid=38341519, collection=test_rag_check
16:11:06 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: query=CRISPR-Cas9 genome editing in zebrafish embryonic ..., collection=test_rag_check
16:11:06 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: Using collection_id: a092e3dc-5d93-44d6-a24f-7c9c0a4375ad
16:11:06 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: claim=CRISPR-Cas9 genome editing in zebrafish embryonic ...
16:11:06 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: calling multi_query tool...


Validating weak claim (CRISPR/zebrafish, PMID 38341519)...


16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 4 valid queries
16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: What are the effects of CRISPR-Cas9 genome editing on zebraf...
16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: Which CRISPR-Cas9 strategies (knockouts, knockins, or base e...
16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: Which zebrafish genes involved in neural crest development a...
16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: What are the methodological considerations (timing, delivery...
16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 4 queries
16:11:12 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: searching query 1/4: What are the effects of CRISPR-Cas9 genome editing...
16:11:12 [DEBUG] src.agents.deep_interpreter.t


Result:
  valid: True
  confidence: 0.58
  supporting_text: #### Exosomal lncRNAs

Long non-coding RNAs (lncRNAs) are non-coding RNAs exceeding 200 nucleotides in length and play p...
  source: full_text

Matching confidence: 0.781
Weak confidence: 0.580
  [PASS] validate_weak_lower_confidence


In [15]:
# Test @tool sync wrapper via .invoke() (exercises run_async ThreadPoolExecutor path)
print("Testing @tool sync wrapper: validate_claim_fulltext.invoke()...")
try:
    sync_result = validate_claim_fulltext.invoke({
        "claim": "Spatial transcriptomics reveals heterogeneous tumor microenvironment",
        "pmid": "37820580",
        "part_name": TEST_PART_NAME,
    })
    print(f"  Result: valid={sync_result.get('valid')}, "
          f"confidence={sync_result.get('confidence')}, "
          f"source={sync_result.get('source')}")
    record("tool_sync_wrapper", isinstance(sync_result, dict) and "valid" in sync_result)
except Exception as e:
    print(f"  Error: {type(e).__name__}: {e}")
    record("tool_sync_wrapper", False)

16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: ============================================================
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: validate_claim_fulltext CALLED
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: claim: Spatial transcriptomics reveals heterogeneous tumor microenvironment
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: pmid: 37820580
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: part_name: test_rag_check
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _RAG_STATE['available']: True
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _RAG_STATE['collection_ids']: {'test_rag_check': 'a092e3dc-5d93-44d6-a24f-7c9c0a4375ad'}
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: ============================================================
16:11:14 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: validate

Testing @tool sync wrapper: validate_claim_fulltext.invoke()...


16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: generate_multi_query: generated 5 valid queries
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 1: How does spatial transcriptomics uncover heterogeneity in th...
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 2: What does spatially resolved transcriptomics reveal about in...
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 3: Mapping the tumor microenvironment: how does spatial transcr...
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 4: What insights into tumor microenvironment heterogeneity can ...
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag:   Query 5: Spatially resolved transcriptomics and heterogeneity: deline...
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_fulltext_rag: generated 5 queries
16:11:18 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: 

  Result: valid=True, confidence=0.678, source=full_text
  [PASS] tool_sync_wrapper


## Section 7: Test 4 — State Management

In [16]:
# Test get_rag_status() and is_rag_mode_active()
status = get_rag_status()
print(f"get_rag_status():")
print(f"  available: {status['available']}")
print(f"  collections: {status['collections']}")

active = is_rag_mode_active()
print(f"\nis_rag_mode_active(): {active}")

# After successful prepare, both should be True
record("status_available", status["available"] is True)
record("status_has_collections", len(status["collections"]) > 0)
record("rag_mode_active", active is True)

get_rag_status():
  available: True
  collections: {'test_rag_check': 'a092e3dc-5d93-44d6-a24f-7c9c0a4375ad'}

is_rag_mode_active(): True
  [PASS] status_available
  [PASS] status_has_collections
  [PASS] rag_mode_active


In [17]:
# Test reset_rag_state() and verify it clears everything
# Save current state for restoration
saved_state = {
    "available": _RAG_STATE["available"],
    "collection_ids": dict(_RAG_STATE["collection_ids"]),
}

reset_rag_state()
print("After reset_rag_state():")
print(f"  available: {_RAG_STATE['available']}")
print(f"  collection_ids: {_RAG_STATE['collection_ids']}")
print(f"  is_rag_mode_active: {is_rag_mode_active()}")

record(
    "reset_clears_state",
    _RAG_STATE["available"] is None
    and len(_RAG_STATE["collection_ids"]) == 0
    and not is_rag_mode_active(),
)

# Restore state for remaining tests
_RAG_STATE["available"] = saved_state["available"]
_RAG_STATE["collection_ids"] = saved_state["collection_ids"]
print(f"\nState restored: available={_RAG_STATE['available']}, "
      f"collections={list(_RAG_STATE['collection_ids'].keys())}")

After reset_rag_state():
  available: None
  collection_ids: {}
  is_rag_mode_active: False
  [PASS] reset_clears_state

State restored: available=True, collections=['test_rag_check']


## Section 8: Test 5 — Fallback Behavior

In [18]:
# Force RAG unavailable and verify fallback behavior
saved_available = _RAG_STATE["available"]
_RAG_STATE["available"] = False
print("Forced _RAG_STATE['available'] = False")

# search_literature_fulltext should return empty list
print("\nTesting search_literature_fulltext fallback...")
fallback_search = search_literature_fulltext.invoke({
    "query": "tumor microenvironment",
    "part_name": TEST_PART_NAME,
    "limit": 5,
})
print(f"  search result: {fallback_search} (expected: [])")
record("fallback_search_empty", fallback_search == [])

# validate_claim_fulltext should return source=abstract
print("\nTesting validate_claim_fulltext fallback...")
fallback_validate = validate_claim_fulltext.invoke({
    "claim": "test claim",
    "pmid": "12345678",
    "part_name": TEST_PART_NAME,
})
print(f"  validate result source: {fallback_validate.get('source')} (expected: 'abstract')")
record("fallback_validate_abstract", fallback_validate.get("source") == "abstract")

# Restore
_RAG_STATE["available"] = saved_available
print(f"\nRestored _RAG_STATE['available'] = {saved_available}")

16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: ============================================================
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_literature_fulltext CALLED
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: query: tumor microenvironment
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: part_name: test_rag_check
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: limit: 5
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _RAG_STATE['available']: False
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: _RAG_STATE['collection_ids']: {'test_rag_check': 'a092e3dc-5d93-44d6-a24f-7c9c0a4375ad'}
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: ============================================================
16:11:21 [DEBUG] src.agents.deep_interpreter.tools.literature_rag: search_literature_fulltext: RAG not available, returning

Forced _RAG_STATE['available'] = False

Testing search_literature_fulltext fallback...
  search result: [] (expected: [])
  [PASS] fallback_search_empty

Testing validate_claim_fulltext fallback...
  validate result source: abstract (expected: 'abstract')
  [PASS] fallback_validate_abstract

Restored _RAG_STATE['available'] = True


In [19]:
# Test check_rag_availability_sync cache performance
# After initial check, cached result should return in < 100ms
# Ensure cache is primed
_ = check_rag_availability_sync()

start = time.perf_counter()
for _ in range(100):
    check_rag_availability_sync()
elapsed_ms = (time.perf_counter() - start) * 1000
avg_ms = elapsed_ms / 100

print(f"100x check_rag_availability_sync: {elapsed_ms:.1f} ms total, {avg_ms:.3f} ms avg")
record("cache_performance", avg_ms < 1.0)  # Should be sub-millisecond when cached

100x check_rag_availability_sync: 0.0 ms total, 0.000 ms avg
  [PASS] cache_performance


## Section 9: Cleanup

In [20]:
# Delete test collection from Vector DB
print(f"Deleting test collection: {TEST_COLLECTION_NAME}")

collection_id = _RAG_STATE["collection_ids"].get(TEST_PART_NAME)
if collection_id:
    try:
        tools = await _get_cci_rag_tools()
        delete_tool = None
        for t in tools:
            if hasattr(t, "name") and "delete_collection" in t.name:
                delete_tool = t
                break

        if delete_tool:
            result = await delete_tool.ainvoke({"collection_id": collection_id})
            print(f"  Deleted collection {collection_id}: {result}")
            record("cleanup_collection_deleted", True)
        else:
            print("  delete_collection tool not found")
            record("cleanup_collection_deleted", False)
    except Exception as e:
        print(f"  Cleanup error: {e}")
        record("cleanup_collection_deleted", False)
else:
    print("  No collection_id to delete (may not have been created)")
    record("cleanup_collection_deleted", True)  # Nothing to clean up is OK

Deleting test collection: deep_interpreter_test_rag_check
  Deleted collection a092e3dc-5d93-44d6-a24f-7c9c0a4375ad: [{'type': 'text', 'text': 'Collection a092e3dc-5d93-44d6-a24f-7c9c0a4375ad deleted successfully!', 'id': 'lc_27496d95-a014-4cd3-b133-608bc9daae12'}]
  [PASS] cleanup_collection_deleted


In [21]:
# Reset state, invalidate MCP clients, remove temp directory
reset_rag_state()
invalidate_client("CCI-RAG")
invalidate_client("pubmed-download")
print("RAG state reset, MCP clients invalidated.")

if os.path.exists(TEST_TEMP_DIR):
    shutil.rmtree(TEST_TEMP_DIR)
    print(f"Temp directory removed: {TEST_TEMP_DIR}")

print("Cleanup complete.")

16:11:22 [DEBUG] src.agents.deep_interpreter.tools.mcp_clients: MCP client cache invalidated for 'CCI-RAG'
16:11:22 [DEBUG] src.agents.deep_interpreter.tools.mcp_clients: MCP client cache invalidated for 'pubmed-download'


RAG state reset, MCP clients invalidated.
Temp directory removed: /var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/rag_test_dokvuyah
Cleanup complete.


## Section 10: Summary

In [22]:
# Final summary
print("=" * 60)
print("LITERATURE RAG INTEGRATION TEST RESULTS")
print("=" * 60)

passed = sum(1 for v in TEST_RESULTS.values() if v)
failed = sum(1 for v in TEST_RESULTS.values() if not v)
total = len(TEST_RESULTS)

for name, result in TEST_RESULTS.items():
    status = "PASS" if result else "FAIL"
    print(f"  [{status}] {name}")

print(f"\n{passed}/{total} passed, {failed} failed")

if failed == 0:
    print("\nAll tests passed!")
else:
    print(f"\n{failed} test(s) failed. Check logs above for details.")

assert failed == 0, f"{failed} test(s) failed"

LITERATURE RAG INTEGRATION TEST RESULTS
  [PASS] env_vars_set
  [PASS] rag_infra_available
  [PASS] prepare_rag_mode
  [PASS] prepare_collection_id
  [PASS] prepare_fulltext_count
  [PASS] prepare_papers_count
  [PASS] state_available_true
  [PASS] state_collection_registered
  [PASS] paper_files_exist
  [PASS] search_returns_results
  [PASS] search_result_structure
  [PASS] search_diverse_queries
  [PASS] search_nonexistent_collection
  [PASS] validate_result_structure
  [PASS] validate_confidence_range
  [PASS] validate_weak_lower_confidence
  [PASS] tool_sync_wrapper
  [PASS] status_available
  [PASS] status_has_collections
  [PASS] rag_mode_active
  [PASS] reset_clears_state
  [PASS] fallback_search_empty
  [PASS] fallback_validate_abstract
  [PASS] cache_performance
  [PASS] cleanup_collection_deleted

25/25 passed, 0 failed

All tests passed!
